In [1]:
# ============================================================================
# 01 — FEASIBILITY: is the within-protein design possible?
# ----------------------------------------------------------------------------
# The strongest version of this study asks about the SAME protein twice: once
# where it does its usual thing, once where it does the opposite. The protein
# is then its own control — same name, same fame, same training frequency.
# This only works if enough proteins have BOTH signs. That is what we count.
# ============================================================================
import os, json, hashlib
from pathlib import Path
import pandas as pd, numpy as np
from dotenv import load_dotenv

load_dotenv()                      # reads .env; keys never appear in the code
ROOT = Path.cwd().parent           # notebooks/ -> project root
RAW  = ROOT/"data"/"raw"; RAW.mkdir(parents=True, exist_ok=True)

import omnipath as op

# ---- download and freeze -------------------------------------------------
COMPONENTS = {
    "causal_ppi":  lambda: op.interactions.OmniPath.get(organism="human", genesymbols=True),
    "collectri":   lambda: op.interactions.CollecTRI.get(organism="human", genesymbols=True),
    "kinaseextra": lambda: op.interactions.KinaseExtra.get(organism="human", genesymbols=True),
}
raw = {}
for name, fn in COMPONENTS.items():
    p = RAW/f"{name}.parquet"
    if p.exists():
        raw[name] = pd.read_parquet(p); tag = "cached"
    else:
        raw[name] = fn(); raw[name].to_parquet(p, index=False); tag = "downloaded"
    print(f"  {name:<14}{len(raw[name]):>8,} rows  [{tag}]")

# ---- filter: directed, signed one way, single proteins -------------------
def is_complex(s):
    return (not isinstance(s,str)) or (not s.strip()) or "COMPLEX" in s.upper() or "_" in s

NEEDS_DIR = {"causal_ppi": True, "kinaseextra": True, "collectri": False}
parts = []
for name, d in raw.items():
    d = d.copy()
    if NEEDS_DIR[name]:
        d = d[d.is_directed.astype(bool)]
    stim, inh = d.is_stimulation.astype(bool), d.is_inhibition.astype(bool)
    d = d[(stim ^ inh).values]                       # exactly one sign
    d = d[~d.source_genesymbol.map(is_complex).values]
    d = d[~d.target_genesymbol.map(is_complex).values]
    d = d[~d.source.astype(str).str.upper().str.contains("COMPLEX").values]
    d = d[~d.target.astype(str).str.upper().str.contains("COMPLEX").values].copy()
    d["sign"] = d.is_stimulation.astype(bool).map({True:"stimulates", False:"inhibits"})
    d["n_refs"] = (d.n_references.fillna(0).astype(int) if "n_references" in d.columns
                   else 0)
    d["component"] = name
    parts.append(d[["source_genesymbol","target_genesymbol","sign","n_refs","component"]])

E = pd.concat(parts, ignore_index=True)
print(f"\nsigned edges before dedup : {len(E):,}")

# remove pairs different sources sign both ways, then de-duplicate
conf = E.groupby(["source_genesymbol","target_genesymbol"]).sign.nunique()
bad  = set(conf[conf>1].index)
E["_pk"] = list(zip(E.source_genesymbol, E.target_genesymbol))
E = E[~E._pk.isin(bad)]
E = E.sort_values("n_refs", ascending=False).drop_duplicates("_pk", keep="first")
print(f"sign-conflicted pairs removed: {len(bad):,}")
print(f"usable signed edges          : {len(E):,}")

# ---- THE KEY COUNT -------------------------------------------------------
print("\n" + "="*74)
print("HOW MANY PROTEINS HAVE BOTH ACTIVATING AND INHIBITING TARGETS?")
print("="*74)

def summarise(min_refs, min_per_sign):
    e = E[E.n_refs >= min_refs]
    g = e.groupby(["source_genesymbol","sign"]).size().unstack(fill_value=0)
    for c in ("stimulates","inhibits"):
        if c not in g.columns: g[c] = 0
    ok = g[(g.stimulates >= min_per_sign) & (g.inhibits >= min_per_sign)]
    return len(ok), int(ok.min(axis=1).sum())

print(f"{'min refs/edge':>14}{'min per sign':>14}{'proteins':>11}{'matched pairs':>15}")
print("-"*54)
for mr in (1, 2, 3, 5):
    for mp in (1, 2, 3):
        n_prot, n_pairs = summarise(mr, mp)
        print(f"{mr:>14}{mp:>14}{n_prot:>11,}{n_pairs:>15,}")

print("\n  'matched pairs' = how many (usual, exception) question pairs we could")
print("  build. We need roughly 300. Read across to find a workable setting.")

# ---- how lopsided is each protein? ---------------------------------------
g = E.groupby(["source_genesymbol","sign"]).size().unstack(fill_value=0)
for c in ("stimulates","inhibits"):
    if c not in g.columns: g[c] = 0
g["total"] = g.stimulates + g.inhibits
g["stim_share"] = g.stimulates / g.total
big = g[g.total >= 5]
print("\n" + "="*74)
print("HOW LOPSIDED ARE PROTEINS?  (proteins with >=5 signed targets)")
print("="*74)
print(f"  proteins: {len(big):,}")
print(f"  >=80% one sign : {int(((big.stim_share>=.8)|(big.stim_share<=.2)).sum()):,} "
      f"({((big.stim_share>=.8)|(big.stim_share<=.2)).mean():.0%})")
print(f"  >=90% one sign : {int(((big.stim_share>=.9)|(big.stim_share<=.1)).sum()):,} "
      f"({((big.stim_share>=.9)|(big.stim_share<=.1)).mean():.0%})")
print("\n  Highly lopsided proteins are where the effect should be strongest:")
print("  a strong default, and a rare exception to it.")

print("\n" + "="*74)
print("EXAMPLE PROTEINS FOR THE WITHIN-PROTEIN DESIGN")
print("="*74)
cand = g[(g.total>=6) & (g.stimulates>=1) & (g.inhibits>=1) &
         ((g.stim_share>=.75)|(g.stim_share<=.25))].sort_values("total", ascending=False)
print(f"  candidates: {len(cand):,}\n")
print(f"{'protein':<12}{'stim':>6}{'inhib':>7}{'% stim':>8}   usual -> exception example")
for gene in cand.head(12).index:
    r = cand.loc[gene]
    usual = "stimulates" if r.stim_share>.5 else "inhibits"
    exc   = "inhibits" if usual=="stimulates" else "stimulates"
    ex = E[(E.source_genesymbol==gene) & (E.sign==exc)].sort_values("n_refs", ascending=False)
    t = ex.iloc[0].target_genesymbol if len(ex) else "?"
    print(f"{gene:<12}{int(r.stimulates):>6}{int(r.inhibits):>7}{r.stim_share:>8.0%}"
          f"   usually {usual}, but {exc} {t}")

c:\Users\tauhe\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  causal_ppi      86,189 rows  [downloaded]
  collectri       64,516 rows  [downloaded]
  kinaseextra     25,643 rows  [downloaded]

signed edges before dedup : 70,227
sign-conflicted pairs removed: 111
usable signed edges          : 61,975

HOW MANY PROTEINS HAVE BOTH ACTIVATING AND INHIBITING TARGETS?
 min refs/edge  min per sign   proteins  matched pairs
------------------------------------------------------
             1             1      1,511          8,577
             1             2        891          7,957
             1             3        630          7,435
             2             1        750          2,927
             2             2        388          2,565
             2             3        266          2,321
             3             1        505          1,642
             3             2        248          1,385
             3             3        153          1,195
             5             1        288            766
             5             2       

In [ ]:
# ============================================================================
# 02 — CAN WE BALANCE ACROSS PROTEIN TYPES AND DATA SOURCES?
# ----------------------------------------------------------------------------
# The example proteins from cell 01 were ALL transcription factors. If the whole
# benchmark is TFs, the finding is about gene regulation, not signalling.
# Here we count how many usable proteins are kinases, TFs, or other, and how
# much each OmniPath component contributes.
# ============================================================================
import urllib.request, urllib.parse, time, json
from pathlib import Path
import pandas as pd, numpy as np

ROOT = Path.cwd().parent
META = ROOT/"data"/"processed"; META.mkdir(parents=True, exist_ok=True)

# ---- which proteins are eligible at each strictness setting ---------------
def eligible(min_refs, min_per_sign):
    e = E[E.n_refs >= min_refs]
    g = e.groupby(["source_genesymbol","sign"]).size().unstack(fill_value=0)
    for c in ("stimulates","inhibits"):
        if c not in g.columns: g[c] = 0
    g["total"] = g.stimulates + g.inhibits
    g["stim_share"] = g.stimulates / g.total
    return g[(g.stimulates >= min_per_sign) & (g.inhibits >= min_per_sign)]

# working setting: >=2 refs per edge, >=2 targets of each sign
G = eligible(2, 2)
print(f"eligible proteins at (refs>=2, >=2 per sign): {len(G):,}")

# ---- where do their edges come from? -------------------------------------
sub = E[(E.n_refs >= 2) & E.source_genesymbol.isin(G.index)]
print("\nedges by component (for eligible proteins only):")
print("  " + sub.component.value_counts().to_string().replace("\n","\n  "))
print("\nproteins whose edges come ONLY from CollecTRI (pure TF):")
only_tf = sub.groupby("source_genesymbol").component.apply(
    lambda s: set(s) == {"collectri"})
print(f"  {int(only_tf.sum()):,} of {len(G):,} ({only_tf.mean():.0%})")

# ---- label each protein by function using UniProt --------------------------
acc = {}
for name in ["causal_ppi","collectri","kinaseextra"]:
    d = raw[name]
    for s, a in zip(d.source_genesymbol, d.source):
        if isinstance(a, str) and "COMPLEX" not in a.upper():
            acc.setdefault(s, a)

need = [acc[g] for g in G.index if g in acc]
print(f"\nfetching UniProt annotation for {len(need):,} proteins "
      f"(~{len(need)/50*1.2/60:.0f} min)")

cache = META/"protein_class.json"
if cache.exists():
    UP = json.loads(cache.read_text()); print("  cached")
else:
    UP = {}
    for i in range(0, len(need), 50):
        q = "(" + " OR ".join(f"accession:{a}" for a in need[i:i+50]) + ")"
        try:
            u = ("https://rest.uniprot.org/uniprotkb/stream?format=tsv"
                 "&fields=accession,protein_name,keyword,ec"
                 f"&query={urllib.parse.quote(q)}")
            txt = urllib.request.urlopen(u, timeout=120).read().decode()
            hdr = txt.splitlines()[0].split("\t")
            for ln in txt.splitlines()[1:]:
                p = ln.split("\t")
                if len(p) >= 2: UP[p[0]] = dict(zip(hdr, p))
        except Exception as ex:
            print(f"  batch {i} failed: {type(ex).__name__}")
        time.sleep(0.2)
    cache.write_text(json.dumps(UP)); print(f"  fetched {len(UP):,}")

def classify(rec):
    if not rec: return "unknown"
    kw = str(rec.get("Keywords","")).lower()
    nm = str(rec.get("Protein names","")).lower()
    ec = str(rec.get("EC number","")); blob = kw + " | " + nm
    if "ubl conjugation pathway" in kw or "ubiquitin-protein ligase" in nm or ec.startswith("2.3.2."): return "E3 ligase"
    if "protein phosphatase" in blob or ec.startswith("3.1.3."): return "phosphatase"
    if "kinase" in blob or ec.startswith(("2.7.1","2.7.10","2.7.11","2.7.12")): return "kinase"
    if "protease" in blob or "peptidase" in blob or ec.startswith("3.4."): return "protease"
    if any(k in kw for k in ["activator","repressor","dna-binding","transcription"]): return "transcription factor"
    return "other"

G = G.copy()
G["pclass"] = [classify(UP.get(acc.get(g))) for g in G.index]

print("\n" + "="*72); print("ELIGIBLE PROTEINS BY FUNCTION"); print("="*72)
vc = G.pclass.value_counts()
for k, v in vc.items():
    print(f"  {k:<22}{v:>6,}  ({v/len(G):>5.1%})")

print("\n  how many matched pairs could each class supply?")
print(f"  {'class':<22}{'proteins':>10}{'pairs':>10}{'median % usual':>16}")
print("  " + "-"*58)
for k in vc.index:
    s = G[G.pclass==k]
    pairs = int(s[["stimulates","inhibits"]].min(axis=1).sum())
    lop = np.maximum(s.stim_share, 1-s.stim_share).median()
    print(f"  {k:<22}{len(s):>10,}{pairs:>10,}{lop:>15.0%}")

print("\n" + "="*72); print("IS A BALANCED DESIGN POSSIBLE?"); print("="*72)
TARGET = 300
usable = vc[vc >= 15].index.tolist()
print(f"  classes with >=15 eligible proteins: {usable}")
per = TARGET // max(len(usable), 1)
print(f"  ~{per} pairs per class would give {per*len(usable)} total")
short = [k for k in usable
         if int(G[G.pclass==k][["stimulates","inhibits"]].min(axis=1).sum()) < per]
print(f"  classes that could NOT supply {per} pairs: {short if short else 'none'}")

print("\n  non-TF examples (these are what the benchmark needs):")
nt = G[(G.pclass.isin(["kinase","phosphatase","E3 ligase","protease"])) &
       (G.total>=6)].sort_values("total", ascending=False)
print(f"\n  {'protein':<12}{'class':<20}{'stim':>6}{'inhib':>7}{'% usual':>9}")
for gene in nt.head(15).index:
    r = nt.loc[gene]
    print(f"  {gene:<12}{r.pclass:<20}{int(r.stimulates):>6}{int(r.inhibits):>7}"
          f"{max(r.stim_share,1-r.stim_share):>9.0%}")

G.to_csv(META/"eligible_proteins.csv")
print(f"\nwritten: {META/'eligible_proteins.csv'}")

eligible proteins at (refs>=2, >=2 per sign): 388

edges by component (for eligible proteins only):
  component
  collectri      6994
  causal_ppi     3065
  kinaseextra    1815

proteins whose edges come ONLY from CollecTRI (pure TF):
  96 of 388 (25%)

fetching UniProt annotation for 388 proteins (~0 min)
